### IMPORT


In [2]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import fashion_mnist
import tensorflow as tf

In [4]:
### Semilla
SEED = 42
tf.random.set_seed(SEED)


### Nº de neuronas


In [3]:
# Nº de neuronas
NNEURONAS= [
    [64, 32, 16],
    [128, 64, 32],
    [32, 32, 16],
    [64, 64, 32]
    ]

FUNCIONES= [
    ["relu", "tanh", "relu"],
    ["relu", "relu", "tanh"],
    ["tanh", "relu", "tanh"],
    ["relu", "tanh", "sigmoid"]
    ]

LEARNING_RATE= 0.001
DROPOUT_RATE = 0.25
LOSS= "mean_squared_error"
EPOCHS= 200
BATCH= 32

In [5]:
# MRE definido
def mean_relative_error(y_true, y_pred):
    epsilon = 1e-7  # para evitar división por cero
    relative_error = tf.abs((y_true - y_pred) / (y_true + epsilon))
    return tf.reduce_mean(relative_error)

In [7]:
# EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',   # Se vigila la pérdida de validación
    patience=10,          # Nº de épocas sin mejora antes de parar
    restore_best_weights=True  # Recupera los mejores pesos
)

### DATASET CALIFORNIA HOUSING


In [8]:
print(" DATASET: California Housing\n")

housing_data = fetch_california_housing()

X = housing_data.data
y = housing_data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

resultados = []
for neuronas,funciones in zip(NNEURONAS,FUNCIONES):
    model= Sequential([
        Input(shape=(8,)),
        Dense(neuronas[0],funciones[0]),
        Dropout(DROPOUT_RATE),
        Dense(neuronas[1],funciones[1]),
        Dropout(DROPOUT_RATE),
        Dense(neuronas[2],funciones[2]),
        Dense(1,activation="linear")
    ])

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss= LOSS,
        metrics=["mae", mean_relative_error])
    
    model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.2,
        epochs=EPOCHS,
        batch_size=BATCH,
        callbacks=[early_stop],
        verbose=0) ### verbose nos comenta lo que vamos realizando

    _, mae, mre = model.evaluate(X_test_scaled, y_test, verbose=0)

    resultados.append({
        "neuronas": neuronas,
        "funciones": funciones,
        "MAE": mae,
        "MRE": mre
    })


resultados_ordenados = sorted(resultados, key=lambda x: x["MRE"])

print("\n📈 Resultados ordenados por mejor MRE:\n")
for r in resultados_ordenados:
    print(f"MRE: {r['MRE']:.4f} | MAE: {r['MAE']:.4f} CONFIG -> Neuronas: {r['neuronas']} Funciones: {r['funciones']}")

 DATASET: California Housing


📈 Resultados ordenados por mejor MRE:

MRE: 0.7287 | MAE: 0.4919 CONFIG -> Neuronas: [64, 64, 32] Funciones: ['relu', 'tanh', 'sigmoid']
MRE: 0.7308 | MAE: 0.5128 CONFIG -> Neuronas: [32, 32, 16] Funciones: ['tanh', 'relu', 'tanh']
MRE: 0.7451 | MAE: 0.4581 CONFIG -> Neuronas: [128, 64, 32] Funciones: ['relu', 'relu', 'tanh']
MRE: 0.7459 | MAE: 0.3585 CONFIG -> Neuronas: [64, 32, 16] Funciones: ['relu', 'tanh', 'relu']


###

En el experimento se evaluaron distintas arquitecturas de red neuronal variando el número de neuronas por capa y las funciones de activación. Los modelos fueron comparados utilizando dos métricas principales: MRE (Mean Relative Error) y MAE (Mean Absolute Error).
Al ordenar por MRE, la mejor configuración fue la formada por [64, 64, 32] neuronas con activaciones [relu, tanh, sigmoid], obteniendo un MRE ≈ 0,7287, el valor más bajo entre todas las pruebas. No obstante, al analizar el MAE, se observa que la arquitectura [64, 32, 16] con activaciones [relu, tanh, relu] logra el MAE más reducido (≈ 0,3585), manteniendo un MRE similar al del resto de modelos.
Dado que las diferencias en MRE son relativamente pequeñas entre configuraciones y la reducción en MAE es significativamente mayor, puede concluirse que la arquitectura [64, 32, 16] ofrece el mejor equilibrio entre precisión del modelo y complejidad computacional. Por ello, se considera la opción más adecuada dentro del conjunto de experimentos realizados.

### FASHION_MNIST

In [9]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

X_train_scaled = X_train.reshape(-1, 28*28).astype("float32") / 255.0
X_test_scaled = X_test.reshape(-1, 28*28).astype("float32") / 255.0

resultados = []

for neuronas, funciones in zip(NNEURONAS, FUNCIONES):
    model = Sequential([
        Input(shape=(X_train_scaled.shape[1],)),
        Dense(neuronas[0], activation=funciones[0]),
        Dropout(DROPOUT_RATE),
        Dense(neuronas[1], activation=funciones[1]),
        Dropout(DROPOUT_RATE),
        Dense(neuronas[2], activation=funciones[2]),
        Dense(10, activation="softmax")  # salida para 10 clases
    ])

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=EPOCHS,
        batch_size=BATCH,
        callbacks=[early_stop],
        verbose=0
    )

    _, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

    resultados.append({
        "neuronas": neuronas,
        "funciones": funciones,
        "accuracy": accuracy
    })

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
# Ordenar por accuracy descendente
resultados_ordenados = sorted(resultados, key=lambda x: x["accuracy"], reverse=True)

print("\n📈 Resultados ordenados por mejor Accuracy:\n")
for r in resultados_ordenados:
    print(f"Accuracy: {r['accuracy']:.4f} CONFIG -> Neuronas: {r['neuronas']} Funciones: {r['funciones']}")


📈 Resultados ordenados por mejor Accuracy:

Accuracy: 0.8289 CONFIG -> Neuronas: [64, 32, 16] Funciones: ['relu', 'tanh', 'relu']
Accuracy: 0.8273 CONFIG -> Neuronas: [128, 64, 32] Funciones: ['relu', 'relu', 'tanh']
Accuracy: 0.8270 CONFIG -> Neuronas: [64, 64, 32] Funciones: ['relu', 'tanh', 'sigmoid']
Accuracy: 0.8142 CONFIG -> Neuronas: [32, 32, 16] Funciones: ['tanh', 'relu', 'tanh']


###

La mejor arquitectura para Fashion-MNIST fue [64, 32, 16] con activaciones relu–tanh–relu, alcanzando el mayor Accuracy (0.8289). Los modelos más grandes, como [128, 64, 32], no mejoraron el rendimiento, lo que indica que aumentar la complejidad no aporta beneficios en este caso. Las arquitecturas medianas y pequeñas obtuvieron resultados muy similares entre sí, con diferencias menores al 1%. El peor desempeño corresponde a la red [32, 32, 16], probablemente por capacidad limitada. En conjunto, el modelo [64, 32, 16] ofrece el mejor equilibrio entre precisión y simplicidad computacional.